In [1]:
import torch
import sys
import os.path as osp
import os
import sys
import numpy as np

sys.path.append("/afs/cern.ch/work/a/adevita/public/Tracking_DC")
from src.dataset.dataset import SimpleIterDataset
from src.utils.train_utils import to_filelist
from torch.utils.data import DataLoader
import dgl  # CPU only version for now
from tqdm import tqdm
from torch_scatter import scatter_sum
import matplotlib.pyplot as plt
import pickle
import numpy as np
import mplhep as hep


hep.style.use("CMS")
import matplotlib
matplotlib.rc('font', size=13)

import plotly
import plotly.graph_objects as go
import plotly.offline as pyo
from plotly.subplots import make_subplots

from src.layers.batch_operations import graph_batch_func

/afs/cern.ch/work/a/adevita/miniconda3/envs/testONNX/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class Args:
    def __init__(self, datasets):
        self.data_train = [datasets]
        self.data_val = [datasets]
        #self.data_train = files_train
        self.data_config = '/afs/cern.ch/work/a/adevita/public/Tracking_DC/config_files/config_tracking_global_vector_newFlag.yaml'
        self.extra_selection = None
        self.train_val_split = 0.8
        self.data_fraction = 1
        self.file_fraction = 1
        self.fetch_by_files = False
        self.fetch_step = 1
        self.steps_per_epoch = None
        self.in_memory = False
        self.local_rank = None
        self.copy_inputs = False
        self.no_remake_weights = False
        self.batch_size = 2
        self.num_workers = 0
        self.demo = False
        self.laplace = False
        self.diffs = False
        self.class_edges = False

In [3]:
# This block is the same as 1_dataset.ipynb
vector = []
hit_type = []
pos_hits_xyz = []

#"/eos/experiment/fcc/ee/datasets/DC_tracking/Pythia_evaluation/Zcard/reco_Zcard_1.root"
datasets = {
    "test": "/eos/experiment/fcc/users/a/adevita/idea_v3_o1_dataset/Pythia/Zcard/Zcard_graphs_300.root",
}

args = {key: Args(value) for key, value in datasets.items()}

datas = {}
files_dict = {}
for key in datasets:
    train_range = (0, args[key].train_val_split)
    train_file_dict, train_files = to_filelist(args[key], 'val')
    
    test_data = SimpleIterDataset(train_file_dict, args[key].data_config, for_training=False,
                                   extra_selection=args[key].extra_selection,
                                   remake_weights=True,
                                   load_range_and_fraction=(train_range, args[key].data_fraction),
                                   file_fraction=args[key].file_fraction,
                                   fetch_by_files=args[key].fetch_by_files,
                                   fetch_step=args[key].fetch_step,
                                   infinity_mode=False,
                                   in_memory=args[key].in_memory,
                                   async_load=False,
                                   name='test')
    
    test_loader = DataLoader(
            test_data,
            num_workers=0,
            batch_size=1,
            drop_last=False,
            pin_memory=True,
            collate_fn=graph_batch_func,
    )
    
    for i, (graph, feats) in enumerate(test_loader):
        print(graph.ndata.keys())
        
        vector = graph.ndata["vector"]        
        hit_type = graph.ndata["hit_type"].unsqueeze(1)
        pos_hits_xyz = graph.ndata["pos_hits_xyz"] 
        # print("vector shape:", vector.shape)
        # print("hit_type shape:", hit_type.shape)
        # print("pos_hits_xyz shape:", pos_hits_xyz.shape)
        
        
        
        

        break 


    
    

=== Restarting DataIter test, seed=None ===
dict_keys(['vector', 'hit_type', 'particle_number', 'particle_number_nomap', 'pos_hits_xyz', 'cellid', 'unique_id', 'is_overlay'])


/afs/cern.ch/work/a/adevita/public/Tracking_DC/src/dataset/functions_graph_tracking_newFlag.py:201: UserWarning:

To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).

/afs/cern.ch/work/a/adevita/miniconda3/envs/testONNX/lib/python3.10/site-packages/dgl/heterograph.py:92: DGLWarning:

Recommend creating graphs by `dgl.graph(data)` instead of `dgl.DGLGraph(data)`.



In [ ]:
def find_condpoints(betas: torch.Tensor, unassigned: torch.Tensor, tbeta: float) -> torch.Tensor:
    n_points = unassigned.size(0)
    size_b = betas.size(0)

    select_condpoints = betas > tbeta
    mask_unassigned = torch.zeros(size_b, dtype=torch.bool)

    for i in range(n_points - 1):
        ii = unassigned[i]
        mask_unassigned[ii.item()] = True

    select_condpoints = select_condpoints & mask_unassigned
    indices_condpoints = torch.nonzero(select_condpoints, as_tuple=False).squeeze()

    if indices_condpoints.numel() == 0:
        return torch.empty(0, dtype=torch.long)

    betas_condpoints = -betas[indices_condpoints]
    sorted_indices = torch.argsort(betas_condpoints, dim=0, descending=False)
    indices_condpoints = indices_condpoints[sorted_indices]

    return indices_condpoints

def get_clustering(output_vector: list[float], num_rows: int, tbeta: float, td: float) -> torch.Tensor:
    output_model_tensor = torch.tensor(output_vector, dtype=torch.float32).reshape(num_rows, 4)
    betas = output_model_tensor[:, 3]
    X = output_model_tensor[:, :3]
    n_points = betas.size(0)

    clustering = torch.zeros(n_points, dtype=torch.long)
    unassigned = torch.arange(n_points, dtype=torch.long)
    indices_condpoints = find_condpoints(betas, unassigned, tbeta)

    index_assignation = 1

    while indices_condpoints.numel() > 0 and unassigned.numel() > 0:
        index_condpoint = indices_condpoints[0]
        d = torch.norm(X[unassigned] - X[index_condpoint], dim=1)
        mask_distance = d < td

        if mask_distance.sum() == 0:
            break

        assigned_to_this_condpoint = unassigned[mask_distance]
        clustering[assigned_to_this_condpoint] = index_assignation

        mask_distance_out = ~mask_distance
        unassigned = unassigned[mask_distance_out]
        indices_condpoints = find_condpoints(betas, unassigned, tbeta)

        index_assignation += 1

    return clustering


In [ ]:


# Set session options
sess_options = ort.SessionOptions()
sess_options.log_severity_level = 0 

# Load ONNX model
session = ort.InferenceSession("/afs/cern.ch/work/a/adevita/public/Tracking_DC/clustering_1.onnx", sess_options)




# Prepare input tensor (torch)
input_data = torch.cat([pos_hits_xyz, hit_type, vector], dim=1)

# Convert torch tensor to numpy as ONNX Runtime expects numpy arrays
input_np = input_data.numpy()

# Get input name for ONNX model (you need this to feed input)
input_name = session.get_inputs()[0].name

# # Run inference
# outputs = session.run(None, {input_name: input_np})

# output_tensor = outputs[0]

# tbeta = 0.6 
# td = 0.3
# num_rows = output_tensor.shape[0]
# output_vector = output_tensor.flatten().tolist() 

# # Call the clustering function
# clustering_result = get_clustering(output_vector, num_rows, tbeta, td)

# print(clustering_result)


2025-07-17 10:01:05.116201836 [I:onnxruntime:, inference_session.cc:594 TraceSessionOptions] Session Options {  execution_mode:0 execution_order:DEFAULT enable_profiling:0 optimized_model_filepath:"" enable_mem_pattern:1 enable_mem_reuse:1 enable_cpu_mem_arena:1 profile_file_prefix:onnxruntime_profile_ session_logid: session_log_severity_level:0 session_log_verbosity_level:0 max_num_graph_transformation_steps:10 graph_optimization_level:3 intra_op_param:OrtThreadPoolParams { thread_pool_size: 0 auto_set_affinity: 0 allow_spinning: 1 dynamic_block_base_: 0 stack_size: 0 affinity_str:  set_denormal_as_zero: 0 } inter_op_param:OrtThreadPoolParams { thread_pool_size: 0 auto_set_affinity: 0 allow_spinning: 1 dynamic_block_base_: 0 stack_size: 0 affinity_str:  set_denormal_as_zero: 0 } use_per_session_threads:1 thread_pool_allow_spinning:1 use_deterministic_compute:0 ep_selection_policy:0 config_options: {  } }
2025-07-17 10:01:05.116231816 [I:onnxruntime:, inference_session.cc:414 operator(

KeyboardInterrupt: 